# NFXP Dynamic Demand — Aguirregabiria (2022): Modelsammenligning

**Monte Carlo sammenligning af to modeller:**

- **Model 1 (Original):** $\beta^{\rm dep}(\ell)$ er brand-specifik (ignorerer købt mængde)
- **Model 2 (Quantity-Dependent):** $\beta^{\rm dep}(\ell, q)$ afhænger af sidst-købte mængde $q\in\{1,2\}$

DGP genereres fra **Model 2**. Model 1 er misspecificeret og kan ikke adskille en forbruger med $q=1$ fra en med $q=2$ i tilstanden — det giver systematisk bias i $\beta^{\rm dep}$.

> Aguirregabiria, V. (2022). *Dynamic demand for differentiated products with fixed-effects unobserved heterogeneity*. Econometrics Journal.

---

### Tilstandsrum

| | Model 1 | Model 2 |
|---|---|---|
| **Tilstand** | $(\ell, d, e)$ | $(\ell,\, q_{\rm last},\, d,\, e)$ |
| **Valg** | $\{0,\,1,\,2\}$ | $\{0,\,(j,1),\,(j,2)\}$ = 5 muligheder |
| **$\beta^{\rm dep}$** | $\beta^{\rm dep}(\ell)\cdot d$ | $\beta^{\rm dep}(\ell, q_{\rm last})\cdot d$ |

Den quantity-afhængige afskrivning: jo mere du købte sidst ($q=2$), jo lavere er din daglige afskrivningsrate (du har mere lager tilbage).

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Set global seed for reproducibility
SEED = 2024
rng_global = np.random.default_rng(SEED)

## 1. Primitives and true parameters

All model constants are defined here — one place to change when experimenting with the calibration.

In [ ]:
# ── Model dimensions ──────────────────────────────────────────────────────────
J     = 2      # brands
T     = 52     # periods per consumer
N     = 2_000  # consumers per panel
D_MAX = 3      # duration cap (d ∈ {0,1,2,3} in array = d ∈ {1,2,3,4} in paper)
DELTA = 0.95   # discount factor
Q_MAX = 2      # max purchase quantity

# ── Unit prices (single price per brand — quantity pays q × p(j)) ─────────────
# No separate PACK2_BASE: buying q units costs γ·q·p(j) per the paper's spec.
BASE_PRICES = np.array([7.0, 10.0])   # unit prices
PROMO_DISC  = np.array([1.5,  2.0])   # promotion discount
PROMO_ENTRY   = 0.18
PROMO_PERSIST = 0.62

# ── True structural parameters (DGP = Model 2) ────────────────────────────────
# Utility follows the paper's specification (eq. 1):
#   u(0)   = α(ℓ)  − β^dep(ℓ, q, d)           [no purchase]
#   u(j,q) = α(j)  − γ·q·p(j) − β^sc(ℓ,j)    [buy brand j, qty q]
#
# α(1) = 0 is the normalisation; γ is the marginal disutility of expenditure.
ALPHA_TRUE   = np.array([0.0,  0.50])
GAMMA_TRUE   = 0.05                          # linear expenditure sensitivity
BETA_SC_TRUE = np.array([[0.00, 0.30],
                          [0.25, 0.00]])

# β^dep(ℓ, q, d) = BETA_DEP_TRUE_M2[ℓ, q] × d   (linear in d, qty-specific rate)
# Shape (J, Q_MAX): row = brand, col = last qty (0=qty1, 1=qty2)
# Buying more units → more stock → lower depreciation rate
BETA_DEP_TRUE_M2 = np.array([[0.35, 0.20],   # brand 1: [qty1, qty2]
                               [0.35, 0.20]])   # brand 2: [qty1, qty2]

# ── Monte Carlo settings ──────────────────────────────────────────────────────
MC_REPS = 20
MC_SEED = 2024

# ── Parameter name lists ──────────────────────────────────────────────────────
PARAM_NAMES_M2 = ["alpha_2", "gamma", "beta_sc_12", "beta_sc_21",
                  "bdep_1_q1", "bdep_1_q2", "bdep_2_q1", "bdep_2_q2"]
THETA_TRUE_M2  = np.array([
    ALPHA_TRUE[1], GAMMA_TRUE,
    BETA_SC_TRUE[0,1], BETA_SC_TRUE[1,0],
    BETA_DEP_TRUE_M2[0,0], BETA_DEP_TRUE_M2[0,1],
    BETA_DEP_TRUE_M2[1,0], BETA_DEP_TRUE_M2[1,1],
])

# Model 1 reference = brand-specific β^dep (mean across quantities)
PARAM_NAMES_M1 = ["alpha_2", "gamma", "beta_sc_12", "beta_sc_21",
                  "beta_dep_1", "beta_dep_2"]
BETA_DEP_REF   = BETA_DEP_TRUE_M2.mean(axis=1)   # [0.275, 0.275]
THETA_REF_M1   = np.array([
    ALPHA_TRUE[1], GAMMA_TRUE,
    BETA_SC_TRUE[0,1], BETA_SC_TRUE[1,0],
    BETA_DEP_REF[0], BETA_DEP_REF[1],
])

PARAM_NAMES = PARAM_NAMES_M1    # backward compat
THETA_TRUE  = THETA_REF_M1

N_CHOICES_M2 = 1 + J * Q_MAX   # 5 choices

print("True parameters (Model 2):")
for n, v in zip(PARAM_NAMES_M2, THETA_TRUE_M2):
    print(f"  {n:<14} = {v}")
print(f"\nModel 1 ref β^dep (mean across qty): {BETA_DEP_REF}")
print(f"BASE_PRICES: {BASE_PRICES}  |  Q_MAX={Q_MAX}  |  D_MAX={D_MAX}")

## 2. Price process — Hi-Lo (Assumption 2.1)

The promotion state $e_t \in \{0,1\}^J$ is a binary vector indicating which brands are on sale. With $J=2$ there are $2^2=4$ possible states. Each brand follows an independent two-state Markov chain, so the joint transition matrix is the outer product of the marginal probabilities.

In [ ]:
promo_states = np.array(
    [[(s >> j) & 1 for j in range(J)] for s in range(2 ** J)], dtype=int
)
N_PROMO = len(promo_states)


def make_promo_transition() -> np.ndarray:
    trans = np.empty((N_PROMO, N_PROMO))
    for s, curr in enumerate(promo_states):
        prob_on = np.where(curr == 1, PROMO_PERSIST, PROMO_ENTRY)
        for sp, nxt in enumerate(promo_states):
            trans[s, sp] = np.prod(np.where(nxt == 1, prob_on, 1.0 - prob_on))
    return trans


PROMO_TRANS = make_promo_transition()

print("Unit prices per promotion state (buying q units costs γ·q·p(j)):")
for s, e in enumerate(promo_states):
    prices = BASE_PRICES - PROMO_DISC * e
    print(f"  State {s}: e={tuple(e)}  unit_price={prices}"
          f"  |  q=1 cost: {prices}  |  q=2 cost: {2*prices}")

## 3. Utility functions (jf. papirets ligning 1)

**Model 2** (quantity-dependent, papirets specifikation):

$$U_{it} = \begin{cases} \alpha(\ell_{it}) - \beta^{\rm dep}(\ell_{it},\, q_{it},\, d_{it}) & \text{hvis } y_{it}=0 \\ \alpha(j) - \gamma\cdot q\cdot p_{it}(j) - \beta^{\rm sc}(\ell_{it},\, j) & \text{hvis } y_{it}=(j,q) \end{cases}$$

$\gamma \cdot q \cdot p_{it}(j)$ er disnytten fra total forbrugsuftgift (lineær i $q$ og $p$). Der er ingen eksplicit indkomstbegrænsning.

Vi parameteriserer $\beta^{\rm dep}(\ell, q, d) = \beta^{\rm dep}[\ell, q] \times d$, dvs. afskrivning er lineær i $d$ med en rate der afhænger af brand og sidst-købt mængde.

**Model 1** (original, Agu 2022 — ignorerer $q$):

$$U_{it} = \begin{cases} \alpha(\ell_{it}) - \beta^{\rm dep}(\ell_{it})\cdot d_{it} & \text{hvis } y_{it}=0 \\ \alpha(j) - \gamma\cdot p_{it}(j) - \beta^{\rm sc}(\ell_{it},\, j) & \text{hvis } y_{it}=j \end{cases}$$

Model 1 er en restriktion af Model 2 med $q=1$ altid og $\beta^{\rm dep}[\ell,q]$ kollapseret til $\beta^{\rm dep}[\ell]$.

In [ ]:
def h(c):
    """Log utility (kept for potential use; not used in paper's linear spec)."""
    return np.log(np.maximum(c, 1e-8))


# ── Model 1: brand-specific β^dep, linear expenditure utility ─────────────────
def flow_util(choice, last_brand, duration, e_idx,
              alpha, gamma, beta_sc, beta_dep) -> float:
    """
    Model 1 deterministic utility (paper eq. 1, q=1 restricted):
      u(0)  = α(ℓ) − β^dep[ℓ] · d
      u(j)  = α(j) − γ · p(j) − β^sc(ℓ, j)

    duration: paper's d ∈ {1, 2, ..., D_MAX+1}  (1 = one period after purchase).
    """
    l      = last_brand - 1
    prices = BASE_PRICES - PROMO_DISC * promo_states[e_idx]
    if choice == 0:
        return alpha[l] - beta_dep[l] * duration
    j = choice - 1
    return alpha[j] - gamma * prices[j] - beta_sc[l, j]


# ── Model 2: quantity-dependent β^dep, total expenditure γ·q·p(j) ─────────────
def flow_util_m2(choice, last_brand, last_qty, duration, e_idx,
                 alpha, gamma, beta_sc, beta_dep) -> float:
    """
    Model 2 deterministic utility (paper eq. 1):
      u(0)     = α(ℓ) − β^dep[ℓ, q_last] · d        [inventory depreciation]
      u((j,q)) = α(j) − γ · q · p(j) − β^sc(ℓ, j)  [total expenditure disutility]

    duration: paper's d ∈ {1, 2, ..., D_MAX+1}  (1 = one period after purchase).
    beta_dep shape: (J, Q_MAX) — rate depends on brand and last purchased quantity.
    """
    l     = last_brand - 1
    q_idx = last_qty - 1
    prices = BASE_PRICES - PROMO_DISC * promo_states[e_idx]

    if choice == 0:
        return alpha[l] - beta_dep[l, q_idx] * duration

    j_idx = (choice - 1) // Q_MAX       # brand index (0-indexed)
    q_buy = (choice - 1) % Q_MAX + 1    # quantity: 1 or 2
    return alpha[j_idx] - gamma * q_buy * prices[j_idx] - beta_sc[l, j_idx]

## 4. Inner loop: Value Function Iteration (VFI)

**Model 1** — tilstandsrum $(\ell,d,e)$, $J\times(D_{\max}+1)\times N_e = 2\times 4\times 4 = 32$ tilstande, $V$ shape `(J, D_MAX+1, N_PROMO)`.

**Model 2** — tilstandsrum $(\ell, q_{\rm last}, d, e)$, $J\times Q_{\max}\times(D_{\max}+1)\times N_e = 2\times 2\times 4\times 4 = 64$ tilstande, $V$ shape `(J, Q_MAX, D_MAX+1, N_PROMO)`.

Den forventede fortsættelsesværdi integreres over den eksogene promotionproces:
$$EV[s] = \sum_{s'} P(s'\mid s)\,V[s']$$

Ved køb nulstilles $d=0$, $\ell$ sættes til ny brand, og $q_{\rm last}$ sættes til ny mængde.

In [ ]:
# ── Model 1 VFI ───────────────────────────────────────────────────────────────
def solve_vfi(alpha, gamma, beta_sc, beta_dep,
              tol: float = 1e-10, max_iter: int = 2_000) -> np.ndarray:
    """
    VFI for Model 1. State (l, d_idx, e). V shape: (J, D_MAX+1, N_PROMO).

    Array index d_idx ∈ {0,...,D_MAX} maps to paper duration d = d_idx+1.
    After purchase → d_idx=0 (paper d=1). No-purchase → d_idx increments by 1.
    """
    V = np.zeros((J, D_MAX + 1, N_PROMO))
    for _ in range(max_iter):
        EV = (V.reshape(J * (D_MAX+1), N_PROMO) @ PROMO_TRANS.T
              ).reshape(J, D_MAX+1, N_PROMO)
        V_new = np.empty_like(V)
        for l_idx in range(J):
            ell = l_idx + 1
            for d in range(D_MAX + 1):
                d_next = min(d + 1, D_MAX)   # array index for no-purchase continuation
                d_dur  = d + 1               # paper's duration value passed to utility
                for e in range(N_PROMO):
                    Q = np.empty(J + 1)
                    Q[0] = (flow_util(0, ell, d_dur, e, alpha, gamma, beta_sc, beta_dep)
                            + DELTA * EV[l_idx, d_next, e])
                    for j_idx in range(J):
                        Q[j_idx+1] = (flow_util(j_idx+1, ell, d_dur, e, alpha, gamma, beta_sc, beta_dep)
                                      + DELTA * EV[j_idx, 0, e])   # purchase → d_idx=0
                    qm = Q.max()
                    V_new[l_idx, d, e] = qm + np.log(np.exp(Q - qm).sum())
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new
    return V


# ── Model 2 VFI ───────────────────────────────────────────────────────────────
def solve_vfi_m2(alpha, gamma, beta_sc, beta_dep,
                 tol: float = 1e-10, max_iter: int = 2_000) -> np.ndarray:
    """
    VFI for Model 2. State (l, q_idx, d_idx, e). V shape: (J, Q_MAX, D_MAX+1, N_PROMO).

    Array index d_idx ∈ {0,...,D_MAX} maps to paper duration d = d_idx+1.
    Transition (paper eq. 2):
      No purchase  → (ℓ, q_last, d+1, e')    [d_idx → min(d_idx+1, D_MAX)]
      Buy (j, q)   → (j,  q,     1,   e')    [d_idx → 0]
    beta_dep shape: (J, Q_MAX).
    """
    V = np.zeros((J, Q_MAX, D_MAX + 1, N_PROMO))
    for _ in range(max_iter):
        EV = (V.reshape(J * Q_MAX * (D_MAX+1), N_PROMO) @ PROMO_TRANS.T
              ).reshape(J, Q_MAX, D_MAX+1, N_PROMO)
        V_new = np.empty_like(V)
        for l_idx in range(J):
            ell = l_idx + 1
            for q_idx in range(Q_MAX):
                q_last = q_idx + 1
                for d in range(D_MAX + 1):
                    d_next = min(d + 1, D_MAX)   # array index for no-purchase continuation
                    d_dur  = d + 1               # paper's duration value passed to utility
                    for e in range(N_PROMO):
                        Q_vals = np.empty(N_CHOICES_M2)
                        # No purchase: state → (l, q_last, d_next, e')
                        Q_vals[0] = (
                            flow_util_m2(0, ell, q_last, d_dur, e, alpha, gamma, beta_sc, beta_dep)
                            + DELTA * EV[l_idx, q_idx, d_next, e]
                        )
                        # Buy (j, q): state → (j, q_buy, d_idx=0, e')  [paper d=1]
                        for j_idx in range(J):
                            for q_buy_idx in range(Q_MAX):
                                c = 1 + j_idx * Q_MAX + q_buy_idx
                                Q_vals[c] = (
                                    flow_util_m2(c, ell, q_last, d_dur, e, alpha, gamma, beta_sc, beta_dep)
                                    + DELTA * EV[j_idx, q_buy_idx, 0, e]
                                )
                        qm = Q_vals.max()
                        V_new[l_idx, q_idx, d, e] = qm + np.log(np.exp(Q_vals - qm).sum())
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new
    return V

## 5. Conditional choice probabilities (CCPs)

CCPs beregnes som multinomial-logit softmax over de valg-specifikke værdier:

$$P(j \mid s;\theta) = \frac{\exp(Q_j)}{\sum_{k}\exp(Q_k)}, \quad Q_j = u(j, s) + \delta\cdot EV(\text{næste tilstand})$$

**Model 1** returnerer `P` med shape `(J, D_MAX+1, N_PROMO, J+1)` — tilstand $(\ell, d, e)$.

**Model 2** returnerer `P` med shape `(J, Q_MAX, D_MAX+1, N_PROMO, N_CHOICES_M2)` — tilstand $(\ell, q_{\rm last}, d, e)$.

De to modeller har **forskelligt tilstandsrum** og brug forskellige simulerings- og LL-funktioner.

In [ ]:
# ── Model 1 CCPs ──────────────────────────────────────────────────────────────
def compute_ccps(V, alpha, gamma, beta_sc, beta_dep) -> np.ndarray:
    """
    P shape: (J, D_MAX+1, N_PROMO, J+1).
    d_idx maps to paper duration d = d_idx+1 (same convention as VFI).
    """
    EV = (V.reshape(J*(D_MAX+1), N_PROMO) @ PROMO_TRANS.T).reshape(J, D_MAX+1, N_PROMO)
    P = np.empty((J, D_MAX+1, N_PROMO, J+1))
    for l_idx in range(J):
        ell = l_idx + 1
        for d in range(D_MAX + 1):
            d_next = min(d + 1, D_MAX)
            d_dur  = d + 1               # paper's d
            for e in range(N_PROMO):
                Q = np.empty(J + 1)
                Q[0] = (flow_util(0, ell, d_dur, e, alpha, gamma, beta_sc, beta_dep)
                        + DELTA * EV[l_idx, d_next, e])
                for j_idx in range(J):
                    Q[j_idx+1] = (flow_util(j_idx+1, ell, d_dur, e, alpha, gamma, beta_sc, beta_dep)
                                  + DELTA * EV[j_idx, 0, e])
                w = np.exp(Q - Q.max())
                P[l_idx, d, e, :] = w / w.sum()
    return P


# ── Model 2 CCPs ──────────────────────────────────────────────────────────────
def compute_ccps_m2(V, alpha, gamma, beta_sc, beta_dep) -> np.ndarray:
    """
    P shape: (J, Q_MAX, D_MAX+1, N_PROMO, N_CHOICES_M2).
    d_idx maps to paper duration d = d_idx+1.
    beta_dep shape: (J, Q_MAX).
    """
    EV = (V.reshape(J * Q_MAX * (D_MAX+1), N_PROMO) @ PROMO_TRANS.T
          ).reshape(J, Q_MAX, D_MAX+1, N_PROMO)
    P = np.empty((J, Q_MAX, D_MAX+1, N_PROMO, N_CHOICES_M2))
    for l_idx in range(J):
        ell = l_idx + 1
        for q_idx in range(Q_MAX):
            q_last = q_idx + 1
            for d in range(D_MAX + 1):
                d_next = min(d + 1, D_MAX)
                d_dur  = d + 1               # paper's d
                for e in range(N_PROMO):
                    Q_vals = np.empty(N_CHOICES_M2)
                    Q_vals[0] = (
                        flow_util_m2(0, ell, q_last, d_dur, e, alpha, gamma, beta_sc, beta_dep)
                        + DELTA * EV[l_idx, q_idx, d_next, e]
                    )
                    for j_idx in range(J):
                        for q_buy_idx in range(Q_MAX):
                            c = 1 + j_idx * Q_MAX + q_buy_idx
                            Q_vals[c] = (
                                flow_util_m2(c, ell, q_last, d_dur, e, alpha, gamma, beta_sc, beta_dep)
                                + DELTA * EV[j_idx, q_buy_idx, 0, e]
                            )
                    w = np.exp(Q_vals - Q_vals.max())
                    P[l_idx, q_idx, d, e, :] = w / w.sum()
    return P

## 6. Datasimulering

`simulate_panel` bruges til Model 1 (original, kun brand-valg).

`simulate_panel_m2` bruges til Model 2 og genererer:
- `Y_choice` ∈ {0,...,4}: fuld valg inkl. mængde (bruges i Model 2-estimation)
- `Y_brand` ∈ {0,1,2}: kun brand (bruges i Model 1-estimation på Model 2-data)
- `Q_LAST`: sidst-købt mængde i tilstanden (bruges kun af Model 2)

`data_m2_to_m1(data_m2)` ekstraherer et Model-1-kompatibelt datasæt fra en Model 2-simulering.

In [ ]:
def _sample_rows(rng, row_probs: np.ndarray) -> np.ndarray:
    """Vectorised categorical draw for N rows simultaneously."""
    u = rng.random(len(row_probs))
    cumsum = np.cumsum(row_probs, axis=1)
    return (u[:, None] > cumsum).sum(axis=1)


# ── Model 1 simulation (original) ─────────────────────────────────────────────
def simulate_panel(P_true: np.ndarray,
                   n_consumers: int = N, n_periods: int = T,
                   seed=None) -> dict:
    """Simulate from Model 1 CCPs. Returns dict {Y, L, D, E_IDX}."""
    rng = np.random.default_rng(seed)
    Y     = np.zeros((n_consumers, n_periods), dtype=int)
    L     = np.zeros((n_consumers, n_periods), dtype=int)
    D     = np.zeros((n_consumers, n_periods), dtype=int)
    E_IDX = np.zeros((n_consumers, n_periods), dtype=int)
    ell   = rng.integers(1, J+1, size=n_consumers)
    dur   = rng.integers(0, D_MAX+1, size=n_consumers)
    e_idx = rng.integers(0, N_PROMO, size=n_consumers)
    for t in range(n_periods):
        L[:, t] = ell; D[:, t] = dur; E_IDX[:, t] = e_idx
        probs = P_true[ell-1, np.minimum(dur, D_MAX), e_idx, :]
        y = _sample_rows(rng, probs)
        Y[:, t] = y
        bought = y > 0
        ell = np.where(bought, y, ell)
        dur = np.where(bought, 0, np.minimum(dur+1, D_MAX))
        e_idx = _sample_rows(rng, PROMO_TRANS[e_idx])
    return {"Y": Y, "L": L, "D": D, "E_IDX": E_IDX}


# ── Model 2 simulation (quantity-dependent) ────────────────────────────────────
def simulate_panel_m2(P_true: np.ndarray,
                      n_consumers: int = N, n_periods: int = T,
                      seed=None) -> dict:
    """
    Simulate from Model 2 CCPs.
    P_true shape: (J, Q_MAX, D_MAX+1, N_PROMO, N_CHOICES_M2).

    Returns dict with:
      Y_choice : full choice ∈ {0,...,N_CHOICES_M2-1}  — for Model 2 estimation
      Y_brand  : brand choice ∈ {0,1,2}                — for Model 1 estimation
      L        : last brand purchased
      Q_LAST   : last quantity purchased (part of Model 2 state)
      D        : duration since last purchase
      E_IDX    : promotion state
    """
    rng = np.random.default_rng(seed)
    Y_choice = np.zeros((n_consumers, n_periods), dtype=int)
    L        = np.zeros((n_consumers, n_periods), dtype=int)
    Q_LAST   = np.ones( (n_consumers, n_periods), dtype=int)
    D        = np.zeros((n_consumers, n_periods), dtype=int)
    E_IDX    = np.zeros((n_consumers, n_periods), dtype=int)

    ell   = rng.integers(1, J+1, size=n_consumers)
    q_lst = np.ones(n_consumers, dtype=int)   # initialise q_last = 1
    dur   = rng.integers(0, D_MAX+1, size=n_consumers)
    e_idx = rng.integers(0, N_PROMO, size=n_consumers)

    for t in range(n_periods):
        L[:, t]      = ell
        Q_LAST[:, t] = q_lst
        D[:, t]      = dur
        E_IDX[:, t]  = e_idx

        # CCP lookup: state (l-1, q_last-1, min(d, D_MAX), e)
        probs = P_true[ell-1, q_lst-1, np.minimum(dur, D_MAX), e_idx, :]
        y = _sample_rows(rng, probs)
        Y_choice[:, t] = y

        # Decode brand and quantity from choice index
        bought       = y > 0
        brand_bought = np.where(bought, (y-1) // Q_MAX + 1, ell)
        qty_bought   = np.where(bought, (y-1) % Q_MAX  + 1, q_lst)

        ell   = brand_bought
        q_lst = qty_bought
        dur   = np.where(bought, 0, np.minimum(dur+1, D_MAX))
        e_idx = _sample_rows(rng, PROMO_TRANS[e_idx])

    # Collapse choice to brand only for Model 1 estimation
    Y_brand = np.where(Y_choice == 0, 0, (Y_choice-1) // Q_MAX + 1)

    return {"Y_choice": Y_choice, "Y_brand": Y_brand,
            "L": L, "Q_LAST": Q_LAST, "D": D, "E_IDX": E_IDX}


def data_m2_to_m1(data_m2: dict) -> dict:
    """Extract Model-1-compatible dict from Model 2 simulation (drops Q_LAST)."""
    return {"Y":     data_m2["Y_brand"],
            "L":     data_m2["L"],
            "D":     data_m2["D"],
            "E_IDX": data_m2["E_IDX"]}

## 7. Log-likelihood og NFXP-objektiv

**Model 1** estimeres på `data_m1` = `{"Y": Y_brand, "L", "D", "E_IDX"}` — brand-valg, ingen mængde.

**Model 2** estimeres på `data_m2` = `{"Y_choice", "L", "Q_LAST", "D", "E_IDX"}` — fuld tilstand.

I MC-studiet genereres data fra Model 2, og begge modeller estimeres på det **samme** panel:
- Model 1 ser kun `Y_brand` og ignorerer `Q_LAST` fra tilstanden
- Model 2 bruger hele `Y_choice` og `Q_LAST`

Model 1's `beta_dep`-estimater vil være biased fordi den ikke kan adskille forbrugere med $q_{\rm last}=1$ og $q_{\rm last}=2$.

In [ ]:
def log_likelihood(data: dict, P: np.ndarray) -> float:
    """Model 1 log-likelihood. data keys: Y (brand), L, D, E_IDX."""
    Y, L, D, E = data["Y"], data["L"], data["D"], data["E_IDX"]
    probs = P[L-1, D, E, Y]
    return float(np.sum(np.log(np.maximum(probs, 1e-300))))


def log_likelihood_m2(data: dict, P: np.ndarray) -> float:
    """Model 2 log-likelihood. data keys: Y_choice, L, Q_LAST, D, E_IDX."""
    Y = data["Y_choice"]
    L, Q, D, E = data["L"], data["Q_LAST"], data["D"], data["E_IDX"]
    probs = P[L-1, Q-1, D, E, Y]
    return float(np.sum(np.log(np.maximum(probs, 1e-300))))


# ── Model 1 ───────────────────────────────────────────────────────────────────
def unpack(theta: np.ndarray):
    """[alpha_2, gamma, beta_sc_12, beta_sc_21, beta_dep_1, beta_dep_2]"""
    alpha    = np.array([0.0, theta[0]])
    gamma    = float(theta[1])
    beta_sc  = np.array([[0.0, theta[2]], [theta[3], 0.0]])
    beta_dep = np.array([theta[4], theta[5]])
    return alpha, gamma, beta_sc, beta_dep


def nfxp_objective(theta: np.ndarray, data: dict) -> float:
    alpha, gamma, beta_sc, beta_dep = unpack(theta)
    V = solve_vfi(alpha, gamma, beta_sc, beta_dep)
    P = compute_ccps(V, alpha, gamma, beta_sc, beta_dep)
    return -log_likelihood(data, P)


def estimate_nfxp(data: dict, theta0: np.ndarray = None, verbose: bool = False):
    """Estimate Model 1 by NFXP (Nelder-Mead). data must have key 'Y' (brand choice)."""
    if theta0 is None:
        theta0 = np.array([0.1, 0.3, 0.3, 0.3, 0.1, 0.1])
    return minimize(
        fun=nfxp_objective, x0=theta0, args=(data,), method="Nelder-Mead",
        options={"maxiter": 10_000, "xatol": 1e-5, "fatol": 1e-5,
                 "disp": verbose, "adaptive": True},
    )


# ── Model 2 ───────────────────────────────────────────────────────────────────
def unpack_m2(theta: np.ndarray):
    """[alpha_2, gamma, beta_sc_12, beta_sc_21, bdep_1q1, bdep_1q2, bdep_2q1, bdep_2q2]"""
    alpha    = np.array([0.0, theta[0]])
    gamma    = float(theta[1])
    beta_sc  = np.array([[0.0, theta[2]], [theta[3], 0.0]])
    beta_dep = theta[4:8].reshape(J, Q_MAX)   # (J, Q_MAX) matrix
    return alpha, gamma, beta_sc, beta_dep


def nfxp_objective_m2(theta: np.ndarray, data: dict) -> float:
    alpha, gamma, beta_sc, beta_dep = unpack_m2(theta)
    V = solve_vfi_m2(alpha, gamma, beta_sc, beta_dep)
    P = compute_ccps_m2(V, alpha, gamma, beta_sc, beta_dep)
    return -log_likelihood_m2(data, P)


def estimate_nfxp_m2(data: dict, theta0: np.ndarray = None, verbose: bool = False):
    """Estimate Model 2 by NFXP. data must have keys 'Y_choice', 'Q_LAST', etc."""
    if theta0 is None:
        theta0 = np.array([0.1, 0.3, 0.3, 0.3, 0.2, 0.1, 0.2, 0.1])
    return minimize(
        fun=nfxp_objective_m2, x0=theta0, args=(data,), method="Nelder-Mead",
        options={"maxiter": 15_000, "xatol": 1e-5, "fatol": 1e-5,
                 "disp": verbose, "adaptive": True},
    )

## 8. Pilot — enkelt panel, begge modeller

DGP: Model 2 (quantity-dependent beta_dep, jf. papirets ligning 1). Vi estimerer begge modeller og verificerer:

- **Model 2** gendanner $\beta^{\rm dep}(\ell, q)$ korrekt for alle kombinationer af brand og mængde
- **Model 1** estimerer én blandet $\beta^{\rm dep}(\ell)$ per brand — forventet et sted *mellem* de sande værdier 0.35 (q=1) og 0.20 (q=2), men biased afhængigt af den empiriske fordeling af $q_{\rm last}$

In [ ]:
# ── Trin 1: Løs Model 2 ved sande parametre ──────────────────────────────────
print("Løser Model 2 DP ved sande parametre...")
a0, g0, sc0, dep0 = unpack_m2(THETA_TRUE_M2)
V_true = solve_vfi_m2(a0, g0, sc0, dep0)
P_true = compute_ccps_m2(V_true, a0, g0, sc0, dep0)
print("  VFI konvergeret.")

# ── Trin 2: Simulér panel fra Model 2 ────────────────────────────────────────
print(f"Simulerer panel (N={N}, T={T}) fra Model 2...")
data_m2 = simulate_panel_m2(P_true, seed=42)
Y = data_m2["Y_choice"]
print(f"  Ingen køb: {(Y==0).mean():.1%}")
for j in range(1, J+1):
    for q in range(1, Q_MAX+1):
        c = 1 + (j-1)*Q_MAX + (q-1)
        print(f"  Brand {j}, qty {q}: {(Y==c).mean():.1%}")

# ── Trin 3a: Model 1 på misspecificeret data ──────────────────────────────────
print("\nEstimerer Model 1 (misspecificeret)...")
data_m1 = data_m2_to_m1(data_m2)
t0  = time.perf_counter()
res1 = estimate_nfxp(data_m1,
                     theta0=THETA_REF_M1 + np.array([0.05, 0.05, 0.05, 0.05, 0.02, 0.02]))
t1  = time.perf_counter() - t0
print(f"  Tid: {t1:.1f}s  Konvergeret: {res1.success}  Iter: {res1.nit}")

# ── Trin 3b: Model 2 korrekt specificeret ─────────────────────────────────────
print("Estimerer Model 2 (korrekt specificeret)...")
t0  = time.perf_counter()
res2 = estimate_nfxp_m2(data_m2,
                         theta0=THETA_TRUE_M2 + np.array([0.05, 0.05, 0.05, 0.05,
                                                           0.02, 0.01, 0.02, 0.01]))
t2  = time.perf_counter() - t0
print(f"  Tid: {t2:.1f}s  Konvergeret: {res2.success}  Iter: {res2.nit}")

# ── Resultattabel ─────────────────────────────────────────────────────────────
print("\n─── Model 1 (brand-specifik beta_dep — misspecificeret) ───")
print(f"  Referencepunkt: THETA_REF_M1 = {THETA_REF_M1}")
df1 = pd.DataFrame({"Param":   PARAM_NAMES_M1,
                    "Ref.":    THETA_REF_M1,
                    "NFXP":    res1.x,
                    "Afvigelse": res1.x - THETA_REF_M1})
print(df1.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

print("\n─── Model 2 (quantity-dependent beta_dep — korrekt specificeret) ───")
df2 = pd.DataFrame({"Param": PARAM_NAMES_M2,
                    "Sand":  THETA_TRUE_M2,
                    "NFXP":  res2.x,
                    "Bias":  res2.x - THETA_TRUE_M2})
print(df2.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

## 9. Monte Carlo-sammenligning: Model 1 vs. Model 2

For hver replikation:
1. Simulér nyt panel fra Model 2-DGP ($\beta^{\rm dep}(\ell, q)$)
2. Estimér **Model 1** på `Y_brand` — ser ikke mængden, ingen `Q_LAST` i tilstanden
3. Estimér **Model 2** på `Y_choice` + `Q_LAST` — korrekt specificeret

**Forventet resultat:**
- Model 2 → lav RMSE, bias tæt på nul for alle 8 parametre
- Model 1 → `beta_dep_j` estimeres et sted imellem 0.20 og 0.35, fordi den ikke kan adskille $q=1$-forbrugere fra $q=2$-forbrugere

In [ ]:
def run_monte_carlo_comparison(n_reps: int = MC_REPS, seed: int = MC_SEED):
    """
    MC-sammenligning: DGP = Model 2.
    Begge modeller estimeres på det SAMME simulerede panel i hver rep.
    Model 1 er misspecificeret (ignorerer Q_LAST og mængdevalget).
    """
    rng_master = np.random.default_rng(seed)
    rep_seeds  = rng_master.integers(0, 1_000_000, size=n_reps)

    # Løs Model 2 én gang ved sande parametre
    a_t, g_t, sc_t, dep_t = unpack_m2(THETA_TRUE_M2)
    V_true = solve_vfi_m2(a_t, g_t, sc_t, dep_t)
    P_true = compute_ccps_m2(V_true, a_t, g_t, sc_t, dep_t)

    print(f"MC  |  J={J}, T={T}, N={N}, D_MAX={D_MAX}, reps={n_reps}")
    print(f"DGP: Model 2  (bdep q1={BETA_DEP_TRUE_M2[:,0]}, bdep q2={BETA_DEP_TRUE_M2[:,1]})\n")

    rows_m1, rows_m2 = [], []

    for rep in range(1, n_reps + 1):
        data_m2 = simulate_panel_m2(P_true, n_consumers=N, n_periods=T,
                                    seed=int(rep_seeds[rep-1]))
        data_m1 = data_m2_to_m1(data_m2)

        rng_s  = np.random.default_rng(int(rep_seeds[rep-1]) + 999)
        pert6  = rng_s.normal(0, 0.05, size=6)
        theta0_1 = THETA_REF_M1   + pert6
        theta0_2 = THETA_TRUE_M2  + np.append(pert6, [0.01, 0.01])

        t0 = time.perf_counter()
        r1 = estimate_nfxp(data_m1, theta0=theta0_1)
        t1 = time.perf_counter() - t0

        t0 = time.perf_counter()
        r2 = estimate_nfxp_m2(data_m2, theta0=theta0_2)
        t2 = time.perf_counter() - t0

        print(f"  Rep {rep:>3}/{n_reps}  "
              f"M1 konv={r1.success}({t1:>4.0f}s)  "
              f"M2 konv={r2.success}({t2:>4.0f}s)")

        for k, name in enumerate(PARAM_NAMES_M1):
            rows_m1.append({"rep": rep, "param": name,
                            "ref": THETA_REF_M1[k], "estimat": r1.x[k],
                            "bias": r1.x[k] - THETA_REF_M1[k],
                            "kv_fejl": (r1.x[k] - THETA_REF_M1[k])**2,
                            "konv": int(r1.success)})

        for k, name in enumerate(PARAM_NAMES_M2):
            rows_m2.append({"rep": rep, "param": name,
                            "sand": THETA_TRUE_M2[k], "estimat": r2.x[k],
                            "bias": r2.x[k] - THETA_TRUE_M2[k],
                            "kv_fejl": (r2.x[k] - THETA_TRUE_M2[k])**2,
                            "konv": int(r2.success)})

    df_m1 = pd.DataFrame(rows_m1)
    df_m2 = pd.DataFrame(rows_m2)

    def make_summary(df, pnames, ref_vals):
        return pd.DataFrame([{
            "param":     name,
            "ref/sand":  ref_vals[i],
            "mean_est":  df[df["param"]==name]["estimat"].mean(),
            "bias":      df[df["param"]==name]["bias"].mean(),
            "std":       df[df["param"]==name]["estimat"].std(ddof=1),
            "rmse":      np.sqrt(df[df["param"]==name]["kv_fejl"].mean()),
            "konv_rate": df[df["param"]==name]["konv"].mean(),
        } for i, name in enumerate(pnames)])

    sum_m1 = make_summary(df_m1, PARAM_NAMES_M1, THETA_REF_M1)
    sum_m2 = make_summary(df_m2, PARAM_NAMES_M2, THETA_TRUE_M2)
    return df_m1, df_m2, sum_m1, sum_m2


# Kør MC (~40 min ved MC_REPS=20, N=2000 — to NFXP-estimationer pr. rep.)
results_m1, results_m2, summary_m1, summary_m2 = run_monte_carlo_comparison()

## 10. Resultater og visualisering

Tre dele:
1. **Opsummeringstabel** — Mean estimate, bias, std.afv. og RMSE for begge modeller
2. **Boxplot** — Distribution af MC-estimater vs. sande værdier (side om side, M1 vs M2)
3. **RMSE/Bias søjlediagram** — Direkte sammenligning af fejlmål

In [ ]:
fmt = lambda x: f"{x:.4f}"

print("=" * 70)
print("MODEL 1  (brand-specifik beta_dep — misspecificeret, DGP = Model 2)")
print(f"Ref. = gennemsnit af sande q1/q2-rater = {BETA_DEP_REF[0]:.3f}")
print("=" * 70)
print(summary_m1.to_string(index=False, float_format=fmt))

print("\n" + "=" * 70)
print("MODEL 2  (quantity-dependent beta_dep — korrekt specificeret)")
print("=" * 70)
print(summary_m2.to_string(index=False, float_format=fmt))

results_m1.to_csv("nfxp_mc_results_m1.csv", index=False)
results_m2.to_csv("nfxp_mc_results_m2.csv", index=False)
summary_m1.to_csv("nfxp_mc_summary_m1.csv", index=False)
summary_m2.to_csv("nfxp_mc_summary_m2.csv", index=False)
print("\nGemt: nfxp_mc_results_m{1,2}.csv  og  nfxp_mc_summary_m{1,2}.csv")

In [ ]:
latex_lbl = {
    "alpha_2":    r"$\alpha_2$",
    "gamma":      r"$\gamma$",
    "beta_sc_12": r"$\beta^{sc}_{12}$",
    "beta_sc_21": r"$\beta^{sc}_{21}$",
    "beta_dep_1": r"$\beta^{dep}_1$",
    "beta_dep_2": r"$\beta^{dep}_2$",
    "bdep_1_q1":  r"$\beta^{dep}_{1,q1}$",
    "bdep_1_q2":  r"$\beta^{dep}_{1,q2}$",
    "bdep_2_q1":  r"$\beta^{dep}_{2,q1}$",
    "bdep_2_q2":  r"$\beta^{dep}_{2,q2}$",
}

# ── 2×3 boxplot: de 6 fælles parametre, M1 (rød) vs M2 (blå) ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

# For M2 comparison: map M1-name → corresponding M2-name(s) for display
m2_common_names = ["alpha_2", "gamma", "beta_sc_12", "beta_sc_21",
                   "bdep_1_q1", "bdep_2_q1"]   # q1 branch for side-by-side
m2_true_common  = [THETA_TRUE_M2[i] for i in [0,1,2,3,4,6]]
ref_m1_common   = list(THETA_REF_M1)

for ax, (n_m1, n_m2, ref, true_m2) in enumerate(zip(
        PARAM_NAMES_M1, m2_common_names, ref_m1_common, m2_true_common)):

    est_m1 = results_m1[results_m1["param"] == n_m1]["estimat"].to_numpy()
    est_m2 = results_m2[results_m2["param"] == n_m2]["estimat"].to_numpy()

    bp = axes[ax].boxplot(
        [est_m1, est_m2], vert=True, patch_artist=True,
        labels=["M1\n(misspec.)", "M2\n(korrekt)"],
        boxprops=dict(alpha=0.75),
        medianprops=dict(linewidth=2),
    )
    bp["boxes"][0].set_facecolor("salmon")
    bp["boxes"][1].set_facecolor("steelblue")
    bp["medians"][0].set_color("darkred")
    bp["medians"][1].set_color("navy")

    axes[ax].axhline(ref,     color="firebrick", lw=1.6, ls="--", label=f"Ref M1={ref:.2f}")
    axes[ax].axhline(true_m2, color="navy",      lw=1.2, ls=":",  label=f"Sand M2={true_m2:.2f}")
    axes[ax].set_title(latex_lbl.get(n_m1, n_m1), fontsize=12)
    axes[ax].set_ylabel("Estimat", fontsize=8)
    axes[ax].legend(fontsize=7)

fig.suptitle(
    f"Boxplot: Model 1 (rød) vs. Model 2 (blå)  |  J={J}, T={T}, N={N}, reps={MC_REPS}\n"
    r"$\beta^{dep}$-søjlerne viser M2-estimat for $q=1$; M2-kolonne er korrekt specificeret",
    fontsize=10,
)
plt.tight_layout()
plt.savefig("nfxp_mc_boxplots.pdf", bbox_inches="tight")
plt.show()

# ── Ekstra: alle 4 beta_dep-parametre i Model 2 ───────────────────────────────
dep_names_m2 = ["bdep_1_q1", "bdep_1_q2", "bdep_2_q1", "bdep_2_q2"]
fig2, axes2  = plt.subplots(1, 4, figsize=(14, 4))
for ax, name in zip(axes2, dep_names_m2):
    est   = results_m2[results_m2["param"] == name]["estimat"].to_numpy()
    true_v = THETA_TRUE_M2[PARAM_NAMES_M2.index(name)]
    ax.boxplot(est, vert=True, patch_artist=True,
               boxprops=dict(facecolor="steelblue", alpha=0.7),
               medianprops=dict(color="navy", linewidth=2))
    ax.axhline(true_v, color="firebrick", lw=1.8, ls="--", label=f"Sand={true_v:.2f}")
    ax.set_title(latex_lbl[name], fontsize=12)
    ax.set_ylabel("Estimat", fontsize=9)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mticker.NullLocator())

fig2.suptitle(r"Model 2: alle $\beta^{dep}(\ell,q)$-parametre (korrekt specificeret)", fontsize=11)
plt.tight_layout()
plt.savefig("nfxp_mc_boxplots_dep.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── RMSE og |Bias|: M1 vs. M2 på de 6 fælles parametre ──────────────────────
common_m1  = PARAM_NAMES_M1
common_m2  = ["alpha_2", "gamma", "beta_sc_12", "beta_sc_21", "bdep_1_q1", "bdep_2_q1"]
labels     = [latex_lbl.get(n, n) for n in common_m1]

rmse_m1 = [summary_m1.loc[summary_m1["param"]==n, "rmse"].values[0]        for n in common_m1]
rmse_m2 = [summary_m2.loc[summary_m2["param"]==n, "rmse"].values[0]        for n in common_m2]
bias_m1 = [abs(summary_m1.loc[summary_m1["param"]==n, "bias"].values[0])   for n in common_m1]
bias_m2 = [abs(summary_m2.loc[summary_m2["param"]==n, "bias"].values[0])   for n in common_m2]

x     = np.arange(len(common_m1))
w     = 0.20
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, vm1, vm2, ylabel, title in [
    (axes[0], rmse_m1, rmse_m2, "RMSE",   "RMSE pr. parameter"),
    (axes[1], bias_m1, bias_m2, "|Bias|", "|Bias| pr. parameter"),
]:
    ax.bar(x - w, vm1, 2*w, label="Model 1 (misspec.)",
           color="salmon", alpha=0.85, edgecolor="black", lw=0.5)
    ax.bar(x + w, vm2, 2*w, label="Model 2 (korrekt)",
           color="steelblue", alpha=0.85, edgecolor="black", lw=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.axhline(0, color="black", lw=0.5)

bdep_q1 = BETA_DEP_TRUE_M2[0, 0]
bdep_q2 = BETA_DEP_TRUE_M2[0, 1]
fig.suptitle(
    f"Model 1 vs. Model 2 — fejlmål  (reps={MC_REPS}, N={N}, T={T})\n"
    r"$\beta^{dep}$-søjle: M1 vs. M2 q=1-komponent  |  "
    f"DGP: $\\beta^{{dep}}(\\ell,q1)={bdep_q1:.2f},\\ \\beta^{{dep}}(\\ell,q2)={bdep_q2:.2f}$",
    fontsize=9,
)
plt.tight_layout()
plt.savefig("nfxp_mc_bias_rmse.pdf", bbox_inches="tight")
plt.show()